# Dataset Explorer

**Lumexa Python Foundations · Course 3: Python Tools & Data Foundations · Lessons 4–5**

A Jupyter Notebook project that loads, inspects, filters and analyzes a real-world dataset: a table of exoplanets (planets around other stars).

* **Part 1 · Lesson 4:** the numbers of the dataset as NumPy arrays: shapes, masks and vectorized maths.
* **Part 2 · Lesson 5:** the whole table with Pandas: load, inspect, select, filter, sort and count.

▶ Run the code cell below (**Shift + Enter**). It downloads the data from the internet; if that does not work, it uses a small saved sample. To keep your own copy, choose **File → Save a copy in Drive**.

In [ ]:
# ======================================================================
#  DATASET EXPLORER
#  Lumexa Python Foundations · Course 3: Python Tools & Data Foundations
#  Part 1 · Lesson 4: NumPy (the numbers of a real dataset as arrays)
#  Part 2 · Lesson 5: Pandas (load, inspect, select, filter, sort and count)
# ======================================================================
# The data is a table of exoplanets: planets that orbit stars other than our Sun, found by astronomers up to 2014.
#   method          how the planet was discovered
#   number          how many planets are known in that star system
#   orbital_period  how long one orbit takes, in Earth days
#   mass            the planet's mass, in Jupiter masses
#   distance        how far away the star system is, in parsecs (1 parsec is about 3.26 light-years)
#   year            the year of discovery

import csv
import io
import urllib.request

import numpy as np
import pandas as pd

URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/planets.csv"

# A small saved sample, so the project still runs when there is no internet connection.
SAMPLE_CSV = """method,number,orbital_period,mass,distance,year
Pulsar Timing,3,25.262,,,1992
Radial Velocity,2,349.7,1.25,25.64,2001
Pulsar Timing,1,36525.0,,,2003
Radial Velocity,2,937.7,1.24,28.04,2003
Radial Velocity,1,303.0,5.25,92.51,2003
Imaging,1,,,140.0,2004
Radial Velocity,4,4205.8,1.814,15.28,2004
Imaging,1,,,165.0,2007
Radial Velocity,1,714.3,10.6,,2007
Radial Velocity,1,677.8,19.8,,2007
Radial Velocity,1,326.03,19.4,110.62,2007
Transit,1,3.1915239,,174.0,2007
Eclipse Timing Variations,2,3321.0,,130.72,2008
Imaging,1,318280.0,,7.69,2008
Microlensing,1,,,,2008
Radial Velocity,6,9.6184,0.0208,12.83,2008
Radial Velocity,1,103.95,5.76,55.19,2008
Microlensing,1,,,,2009
Radial Velocity,1,9.6737,0.041,27.45,2009
Transit,1,3.54846,,,2009
Microlensing,1,,,,2010
Radial Velocity,1,157.57,1.7,140.85,2011
Radial Velocity,1,1966.1,1.34,48.64,2011
Radial Velocity,2,22.656,0.0322,32.31,2011
Radial Velocity,2,7100.0,1.6,15.1,2011
Radial Velocity,2,2277.0,0.246,38.64,2011
Transit,5,3.6961219,,290.0,2011
Transit,1,5.452654,,257.0,2011
Transit,1,1.855558,,770.0,2011
Transit,2,12.3335,,1200.0,2011
Radial Velocity,6,51.76,0.0164,12.83,2012
Transit,1,3.8559,,480.0,2012
Transit,2,12.883,,,2012
Transit,5,0.74296,,303.0,2012
Transit,2,7.8773565,,,2012
Imaging,1,,,91.57,2013
Transit,1,3.00516,,,2013
Transit,2,114.73635,,,2013
Transit,7,7.008151,,780.0,2013
Transit,2,22.804,,,2013
"""

try:
    with urllib.request.urlopen(URL, timeout=10) as response:
        csv_text = response.read().decode("utf-8")
    if not csv_text.startswith("method,number,orbital_period,mass,distance,year"):
        raise ValueError("this is not the planets file")
    print("Data source: the full dataset, downloaded from the internet.")
except (OSError, ValueError):
    csv_text = SAMPLE_CSV
    print("Data source: a small saved sample (the download did not work, so your numbers will be smaller).")


# ----------------------------------------------------------------------
# PART 1 · Lesson 4: the numbers of the dataset as NumPy arrays
# ----------------------------------------------------------------------
# An array holds numbers only, so Part 1 keeps the planets whose orbital period, mass and distance are all known.
# (Part 2 shows how Pandas deals with the gaps.)
known = [row for row in csv.DictReader(io.StringIO(csv_text))
         if row["orbital_period"] and row["mass"] and row["distance"]]

periods = np.array([float(row["orbital_period"]) for row in known])   # Earth days
masses = np.array([float(row["mass"]) for row in known])              # Jupiter masses
distances = np.array([float(row["distance"]) for row in known])       # parsecs
years = np.array([int(row["year"]) for row in known])                 # year of discovery

print()
print(f"{len(known)} planets have a known orbital period, mass and distance.")
print("Shape:", masses.shape, "| size:", masses.size, "| dtype:", masses.dtype, "| years dtype:", years.dtype)

# Summary functions turn a whole array into one number.
for name, values in (("Orbital period (days)", periods), ("Mass (Jupiters)", masses), ("Distance (parsecs)", distances)):
    print(f"{name}: mean {values.mean():.2f}, median {np.median(values):.2f}, "
          f"smallest {values.min():.2f}, largest {values.max():.2f}")

# Vectorized maths: one line changes every planet at once, with no loop.
period_years = periods / 365.25        # days -> years
distance_ly = distances * 3.26         # parsecs -> light-years
mass_earths = masses * 317.8           # Jupiter masses -> Earth masses
print()
print(f"The longest orbit takes {period_years.max():,.0f} Earth years.")
print(f"The farthest star system is {distance_ly.max():,.0f} light-years away.")
print(f"The heaviest planet is {mass_earths.max():,.0f} times the mass of the Earth.")

heaviest = masses.argmax()             # the position of the largest value
print(f"Position {heaviest} holds the heaviest planet: found in {years[heaviest]}, {distances[heaviest]:.1f} parsecs away.")

# Masks: a condition gives True or False for every planet, and .sum() counts the True values.
heavy = masses > 10
near = distances < 50
recent = years > 2010
print()
print(f"Among these {len(known)} planets:")
print("  heavier than 10 Jupiters:", heavy.sum())
print("  closer than 50 parsecs:", near.sum())
print("  found after 2010:", recent.sum())
print("  close AND found after 2010:", (near & recent).sum())
print("The 5 lightest of the close planets (Jupiter masses):", np.sort(masses[near])[:5])

# Normalise: rescale the masses to the range 0 to 1 (the lightest becomes 0 and the heaviest 1).
normalised = (masses - masses.min()) / (masses.max() - masses.min())
print("Normalised masses go from", normalised.min(), "to", normalised.max())

# A 2D array: 3 rows (period, mass, distance) and one column for each planet.
table = np.array([periods, masses, distances])
print()
print("2D array shape:", table.shape)
print("Average of each row (period, mass, distance):", np.round(table.mean(axis=1), 2))
print("All three measurements of the heaviest planet:", table[:, heaviest])


# ----------------------------------------------------------------------
# PART 2 · Lesson 5: the whole table with Pandas
# ----------------------------------------------------------------------
# A DataFrame is built from NumPy arrays, so the ideas are the same: shapes, masks and vectorized operations.
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

planets = pd.read_csv(io.StringIO(csv_text))     # with a file or a web address you write pd.read_csv("planets.csv")

# Look before you ask: head, shape, columns, info and describe.
print()
print("Rows and columns:", planets.shape)
print("Columns:", list(planets.columns))
print()
print(planets.head(10))
print()
planets.info()
print()
print(planets.describe())

# Which columns have missing values? (Lesson 6 shows what to do about them.)
missing = planets.isna().sum()
print()
for column in missing.index:
    if missing[column] > 0:
        print(f"{column}: {missing[column]} missing ({missing[column] / len(planets) * 100:.0f}% of the rows)")

# Selecting: one column (a Series), several columns (a list inside the brackets) and rows by position.
print()
print(planets["method"].head(3))
print()
print(planets[["method", "year", "distance"]].head(5))
print()
print(planets.iloc[10:15])             # rows 10 to 14: the end is not included

# Filtering: a condition keeps the rows where it is True. Join two with & (and) or | (or), each one in brackets.
print()
print("1. Planets found after 2010:", len(planets[planets["year"] > 2010]))
print("2. Planets heavier than 10 Jupiters:", len(planets[planets["mass"] > 10]),
      f"(Part 1 counted {heavy.sum()}: it only had the {len(known)} planets without gaps)")
print("3. Planets closer than 50 parsecs with a known mass:",
      len(planets[(planets["distance"] < 50) & (planets["mass"].notna())]))
print("4. Radial Velocity planets that orbit in under 10 days:",
      len(planets[(planets["method"] == "Radial Velocity") & (planets["orbital_period"] < 10)]))

# Sorting: sort_values puts the rows in order and gives back a new DataFrame.
print()
print("The 5 longest orbits (Earth days):")
print(planets.sort_values("orbital_period", ascending=False)[["method", "orbital_period", "year"]].head(5))
print()
print("The 5 closest star systems (parsecs):")
print(planets.sort_values("distance")[["method", "distance", "year"]].head(5))

# Counting: value_counts shows how often each value appears, the most common first.
by_method = planets["method"].value_counts()
by_year = planets["year"].value_counts()
print()
print(by_method)

# Findings: what the data says, and what it cannot say.
print()
print("FINDINGS")
print(f"- The table has {len(planets)} planets, found with {len(by_method)} different methods.")
print(f"- The most common method is {by_method.index[0]}: {by_method.iloc[0]} planets, "
      f"{by_method.iloc[0] / len(planets) * 100:.0f}% of the table.")
print(f"- The busiest year was {by_year.index[0]}, with {by_year.iloc[0]} discoveries.")
print(f"- Only {planets['mass'].notna().sum()} of the {len(planets)} planets have a known mass, "
      "so every result about mass uses just a part of the data.")
print(f"- The discoveries run from {planets['year'].min()} to {planets['year'].max()}, "
      "so this table says nothing about the planets found since.")